# Part 3 — Machine Learning

CareLoop Clinics — No-Show Guard


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("appointments_clean.csv")

print(df.shape)
print(df.head())
print(df.columns.tolist())

(990, 16)
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  lead_days  no_show  
0               1   No-Show          1        1  


In [ ]:
# Keep appointment IDs separately for the final risk-score CSV
appointment_ids = df["appointment_id"].copy()

# Select only the approved features
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# Convert Yes/No into 1/0
X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# One-hot encode categorical columns
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

# Target variable
y = df["no_show"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature columns:")
print(X.columns.tolist())

Feature shape: (990, 11)
Target shape: (990,)

Feature columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("No-shows in test set:", y_test.sum())

Training rows: 792
Testing rows: 198
No-shows in test set: 49


In [ ]:
baseline_accuracy = accuracy_score(
    y_test,
    np.zeros(len(y_test), dtype=int)
)

print("Baseline Accuracy:", round(baseline_accuracy, 4))

Baseline Accuracy: 0.7525


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [ ]:
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)

X_test_s = scaler.transform(X_test)

In [ ]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_s, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred = model.predict(X_test_s)

y_prob = model.predict_proba(X_test_s)[:, 1]

In [ ]:
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nPrecision:", round(precision_score(y_test, y_pred), 3))

print("Recall:", round(recall_score(y_test, y_pred), 3))

print("F1 Score:", round(f1_score(y_test, y_pred), 3))

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

Accuracy: 0.788

Confusion Matrix:
[[139  10]
 [ 32  17]]

Precision: 0.63
Recall: 0.347
F1 Score: 0.447
ROC-AUC: 0.789


In [ ]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

print(coefficients.sort_values("Coefficient"))

                    Feature  Coefficient
5             reminder_sent    -0.580669
0                       age    -0.253880
8   booking_channel_Walk-In    -0.186364
6       booking_channel_App    -0.046804
10     visit_type_Follow-up    -0.015658
9   visit_type_Consultation     0.015658
7     booking_channel_Phone     0.215881
1               distance_km     0.253448
2                   fee_inr     0.452761
3                 lead_days     0.614471
4            prior_no_shows     0.649612


In [ ]:
reminder_idx = list(X.columns).index("reminder_sent")

reminder_coef = model.coef_[0][reminder_idx]

print("Reminder coefficient:", reminder_coef)

print("Odds multiplier per 1 SD:",
      np.exp(reminder_coef))

Reminder coefficient: -0.5806686033659922
Odds multiplier per 1 SD: 0.559524141750829


In [ ]:
reminder_sd = scaler.scale_[reminder_idx]

actual_effect = np.exp(reminder_coef / reminder_sd)

print("Reminder standard deviation:", reminder_sd)

print("Odds multiplier for No to Yes:",
      actual_effect)

Reminder standard deviation: 0.4949752621616814
Odds multiplier for No to Yes: 0.3093980903124433


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
dt_full = DecisionTreeClassifier(random_state=42)

dt_full.fit(X_train, y_train)

train_pred_full = dt_full.predict(X_train)
test_pred_full = dt_full.predict(X_test)

print("Full Depth Decision Tree")
print("Training Accuracy:", round(accuracy_score(y_train, train_pred_full), 3))
print("Testing Accuracy:", round(accuracy_score(y_test, test_pred_full), 3))
print("Tree Depth:", dt_full.get_depth())

Full Depth Decision Tree
Training Accuracy: 1.0
Testing Accuracy: 0.672
Tree Depth: 15


In [ ]:
dt_3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

dt_3.fit(X_train, y_train)

train_pred_3 = dt_3.predict(X_train)
test_pred_3 = dt_3.predict(X_test)

print("Depth-3 Decision Tree")
print("Training Accuracy:", round(accuracy_score(y_train, train_pred_3), 3))
print("Testing Accuracy:", round(accuracy_score(y_test, test_pred_3), 3))

Depth-3 Decision Tree
Training Accuracy: 0.822
Testing Accuracy: 0.778


###The full-depth Decision Tree achieves 100% training accuracy but only around 67% test accuracy, indicating overfitting. Restricting the tree to a maximum depth of 3 reduces training accuracy but improves generalisation, with test accuracy around 78%.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
rf_pred = rf.predict(X_test)

rf_prob = rf.predict_proba(X_test)[:, 1]

In [ ]:
print("Random Forest Results")

print("Training Accuracy:",
      round(rf.score(X_train, y_train), 3))

print("Test Accuracy:",
      round(accuracy_score(y_test, rf_pred), 3))

print("Recall:",
      round(recall_score(y_test, rf_pred), 3))

print("ROC-AUC:",
      round(roc_auc_score(y_test, rf_prob), 3))

Random Forest Results
Training Accuracy: 1.0
Test Accuracy: 0.747
Recall: 0.306
ROC-AUC: 0.706


In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance.head(3))

       Feature  Importance
3    lead_days    0.219905
0          age    0.196788
1  distance_km    0.194739


In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, model.predict(X_test_s)),
        accuracy_score(y_test, dt_3.predict(X_test)),
        accuracy_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, model.predict(X_test_s)),
        recall_score(y_test, dt_3.predict(X_test)),
        recall_score(y_test, rf_pred)
    ],
    "ROC_AUC": [
        roc_auc_score(y_test, model.predict_proba(X_test_s)[:, 1]),
        roc_auc_score(y_test, dt_3.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, rf_prob)
    ]
})

print(comparison.round(3))

                     Model  Accuracy  Recall  ROC_AUC
0      Logistic Regression     0.788   0.347    0.789
1  Decision Tree (Depth 3)     0.778   0.224    0.700
2            Random Forest     0.747   0.306    0.706


###Logistic Regression achieved the highest test ROC-AUC among the three models, indicating better ability to distinguish between no-show and attended appointments. Random Forest did not outperform Logistic Regression on this test set. However, the final model selection will be based on cross-validation using training data only.

In [ ]:
from sklearn.model_selection import (
    GridSearchCV,
    cross_val_score,
    cross_val_predict
)

from sklearn.pipeline import Pipeline

In [ ]:
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=5,
    scoring="roc_auc"
)

rf_grid.fit(X_train, y_train)

print("Best Parameters:", rf_grid.best_params_)
print("Best CV ROC-AUC:", round(rf_grid.best_score_, 3))

Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.767


In [ ]:
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
models = {
    "Logistic Regression": lr_pipe,

    "Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "Tuned Random Forest": rf_grid.best_estimator_
}

In [ ]:
cv_results = {}

for name, estimator in models.items():

    scores = cross_val_score(
        estimator,
        X_train,
        y_train,
        cv=5,
        scoring="roc_auc"
    )

    cv_results[name] = scores.mean()

    print(name, ":", round(scores.mean(), 3))

Logistic Regression : 0.774
Decision Tree : 0.678
Random Forest : 0.747
Tuned Random Forest : 0.767


In [ ]:
cv_table = pd.DataFrame({
    "Model": cv_results.keys(),
    "CV_ROC_AUC": cv_results.values()
})

cv_table = cv_table.sort_values(
    by="CV_ROC_AUC",
    ascending=False
)

print(cv_table)

                 Model  CV_ROC_AUC
0  Logistic Regression    0.773714
3  Tuned Random Forest    0.767324
2        Random Forest    0.746602
1        Decision Tree    0.678449


In [ ]:
best_model_name = max(cv_results, key=cv_results.get)

best_model = models[best_model_name]

print("Winning Model:", best_model_name)
print("Winning CV ROC-AUC:", round(cv_results[best_model_name], 3))

Winning Model: Logistic Regression
Winning CV ROC-AUC: 0.774


In [ ]:
oof_probs = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

In [ ]:
thresholds = [0.5, 0.35]

threshold_results = []

for threshold in thresholds:

    oof_pred = (oof_probs >= threshold).astype(int)

    precision = precision_score(y_train, oof_pred)
    recall = recall_score(y_train, oof_pred)
    f1 = f1_score(y_train, oof_pred)

    flagged = oof_pred.sum()

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Flagged": flagged
    })

threshold_table = pd.DataFrame(threshold_results)

print(threshold_table.round(3))

   Threshold  Precision  Recall     F1  Flagged
0       0.50      0.575   0.249  0.347       73
1       0.35      0.513   0.467  0.489      154


In [ ]:
best_threshold = threshold_table.loc[
    threshold_table["F1"].idxmax(),
    "Threshold"
]

print("Selected Threshold:", best_threshold)

Selected Threshold: 0.35


In [ ]:
# Fit the winning model on the complete training set
best_model.fit(X_train, y_train)

# Predict test probabilities
test_probs = best_model.predict_proba(X_test)[:, 1]

# Apply selected threshold
test_pred = (test_probs >= best_threshold).astype(int)

In [ ]:
print("Final Model:", best_model_name)

print("Precision:",
      round(precision_score(y_test, test_pred), 3))

print("Recall:",
      round(recall_score(y_test, test_pred), 3))

print("Number Flagged:",
      int(test_pred.sum()))

print("ROC-AUC:",
      round(roc_auc_score(y_test, test_probs), 3))

Final Model: Logistic Regression
Precision: 0.571
Recall: 0.571
Number Flagged: 49
ROC-AUC: 0.789


In [ ]:
risk_scores = pd.DataFrame({
    "appointment_id": appointment_ids.loc[X_test.index].values,
    "no_show": y_test.values,
    "risk_prob": test_probs,
    "flag": test_pred
})

risk_scores.to_csv(
    "test_risk_scores.csv",
    index=False
)

print(risk_scores.head())
print("\nTotal rows:", len(risk_scores))
print("Flagged appointments:", risk_scores["flag"].sum())

  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0

Total rows: 198
Flagged appointments: 49


In [ ]:
check = pd.read_csv("test_risk_scores.csv")

print(check.shape)
print(check.head())

(198, 4)
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0


###Logistic Regression was selected based on its highest cross-validated ROC-AUC on the training data. A threshold of 0.35 was selected because it achieved a higher F1 score and improved recall compared with the default threshold of 0.5. The clinic can use the risk probabilities to identify appointments requiring follow-up. In a real setting, the threshold would be chosen according to the number of follow-up calls the front desk can make each evening and the cost of leaving appointment slots empty.

In [ ]:
print("Total test records:", len(risk_scores))
print("Total flagged:", risk_scores["flag"].sum())
print("Threshold used:", best_threshold)

print("Missing values:")
print(risk_scores.isnull().sum())

print("Risk probability range:")
print(risk_scores["risk_prob"].min(),
      risk_scores["risk_prob"].max())

print("Flagged count matches predictions:",
      risk_scores["flag"].sum() == test_pred.sum())

Total test records: 198
Total flagged: 49
Threshold used: 0.35
Missing values:
appointment_id    0
no_show           0
risk_prob         0
flag              0
dtype: int64
Risk probability range:
0.01165177623945823 0.9372044248490335
Flagged count matches predictions: True
